In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 32.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 22.6 MB/s eta 0:00:00


In [23]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev2

In [5]:
!pip install -q rwkv huggingface_hub

In [6]:
import os
import gc

import huggingface_hub
import torch

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [26]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    ConversationManager,
    FormatPromptTemplate,
    HistoryManager,
    Message,
    OutputParser,
    PromptRenderer,
)

In [25]:
model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0a4-7.2b-20251208-ctx8192"
# model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

In [12]:
config = GenerationConfig(
    max_new_tokens=500,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

GenerationConfig(max_new_tokens=500, temperature=1.0, top_p=0.3, top_k=0, presence_penalty=0.5, frequency_penalty=0.5, decay_penalty=0.996, stop_tokens=[0, 261])


In [13]:
model = RWKVModel(model_path=model_path)

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/0ca5368635c048d847f17be18454dbef318f2081/rwkv7-g1b-2.9b-20251205-ctx8192 (cuda fp16)



In [27]:
tokenizer = RWKVTokenizer(model)
sampler = RWKVSampler(config)

In [28]:
templates = {
    'system': FormatPromptTemplate('System: {text}\n\n'),
    'user': FormatPromptTemplate('User: {text}{think_mode}\n\n'),
    'assistant': FormatPromptTemplate('Assistant: {think_start}{think_text}{think_end}{text}\n\n'),
}

history = HistoryManager()

mgr = ConversationManager(
    templates=templates,
    history=history,
)

In [29]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [30]:
session.reset()
mgr.build_rwkv7_prompt()

''

In [31]:
mgr.add_system('Include emojis to all your answers.')

In [32]:
mgr.add_user('Tell me a short joke.')
mgr.build_rwkv7_prompt(
    auto_dummy=True
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think'

In [33]:
joke2 = session.generate_message()
print(joke2)

role='assistant' text='\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [34]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\n'

In [35]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think'

In [36]:
joke = session.generate_message()
print(joke)

role='assistant' text='\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [37]:
mgr.add_user('Tell me more complex joke.')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think'

In [39]:
joke3 = session.generate_message()
print(joke3)

role='assistant' text='\n😄 Alright, here’s a slightly more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [40]:
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Alright, here’s a slightly more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\n'

In [41]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Alright, here’s a slightly more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think'

In [43]:
final_answer = session.generate_message()

In [44]:
print(final_answer.metadata['think_text'])

We need to list all jokes told so far. The user wants topics of all jokes that have been told, and explanation of the second joke (the second joke is "Why did the tomato turn red?"). So we need to list all jokes we've told so far. Let's recall:
1. First joke: "What do you call a fake noodle? An impasta." (simple pun)
2. Second joke: "Why did the tomato turn red?" (pun on tomato turning red when cooked or ripe)
3. Third joke: "I'm reading a book on anti-gravity... It's impossible to put down!" (play on words with "impossible" and "put down")
4. Fourth joke: "Why don't scientists trust atoms? Because they make up everything!" (chemistry pun)
5. Fifth joke: "I told my wife she was drawing her eyebrows too high... She looked surprised." (meta-joke about telling a joke)
6. Sixth joke: "Why don't skeletons fight each other? They don't have the guts." (body parts pun)
7. Seventh joke: "What do you call cheese that isn't yours? Nacho cheese!" (food pun)
8. Eighth joke: "I'm reading a book on a

In [45]:
print(final_answer.text)

Below is a quick rundown of every “joke” I’ve shared so far, along with a short explanation of why each one works (or doesn’t).  If you’d like me to add more jokes later, just let me know!
---
### 1️⃣ “What do you call a fake noodle?”  
**Topic:** Word‑play / puns  
**Explanation:** The classic


In [46]:
mgr.build_prompt()

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Alright, here’s a slightly more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes told so far. The user wants topics of all jokes that have been told, and explanation of the second joke (the second joke is "Why did the tomato turn red?"). So we need to list all jokes we\'ve told so far. Let\'

In [47]:
# inspect think
mgr.history.get_message_by_number(8)

Message(role='assistant', text='Below is a quick rundown of every “joke” I’ve shared so far, along with a short explanation of why each one works (or doesn’t).  If you’d like me to add more jokes later, just let me know!\n---\n### 1️⃣ “What do you call a fake noodle?”  \n**Topic:** Word‑play / puns  \n**Explanation:** The classic', metadata={'think_start': '<think>', 'think_text': 'We need to list all jokes told so far. The user wants topics of all jokes that have been told, and explanation of the second joke (the second joke is "Why did the tomato turn red?"). So we need to list all jokes we\'ve told so far. Let\'s recall:\n1. First joke: "What do you call a fake noodle? An impasta." (simple pun)\n2. Second joke: "Why did the tomato turn red?" (pun on tomato turning red when cooked or ripe)\n3. Third joke: "I\'m reading a book on anti-gravity... It\'s impossible to put down!" (play on words with "impossible" and "put down")\n4. Fourth joke: "Why don\'t scientists trust atoms? Because 

In [48]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_rwkv7_prompt(
    auto_dummy=True,
)

'System: Include emojis to all your answers.\n\nUser: Tell me a short joke.\n\nAssistant: <think>\n</think>\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂\n\nUser: Tell me a joke different from previous one.\n\nAssistant: <think>\n</think>\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂\n\nUser: Tell me more complex joke.\n\nAssistant: <think>\n</think>\n😄 Alright, here’s a slightly more complex one for you:  \n**Why did the math book look sad?**  \nBecause it had too many problems and no solutions. 📚😭\n\nUser: List topics of all jokes that you have told me. Explain the second joke. think\n\nAssistant: <think>We need to list all jokes told so far. The user wants topics of all jokes that have been told, and explanation of the second joke (the second joke is "Why did the tomato turn red?"). So we need to list all jokes we\'ve told so far. Let\'

In [50]:
endd = session.generate_message()
print(endd)

role='assistant' text='\nYou’re welcome! 🎉 It was fun to chat and share those quick laughs with you. If you ever want more jokes, riddles, or just a good conversation, feel free to reach out anytime—I’m always here! Have a great day! 😊✨' metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}


In [51]:
print(mgr.history._hash_to_index, mgr.history._messages)
print(session._processed_hash, mgr.history.cumulative_hash)
print(mgr.history.get_message_number(session._processed_hash))
print(session.manager.history.get_message_count())

{0: -1, 940153586: 0, 786751198: 1, 474351376: 2, 4172277263: 3, 2387013392: 4, 1050776399: 5, 1567675728: 6, 3344168371: 7, 3983175888: 8, 926884522: 9, 2896108313: 10} [Message(role='system', text='Include emojis to all your answers.', metadata={}), Message(role='user', text='Tell me a short joke.', metadata={}), Message(role='assistant', text='\n😄 Here’s a quick one for you:  \n**Why did the tomato turn red?**  \nBecause it saw the salad dressing! 🍅😂', metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}), Message(role='user', text='Tell me a joke different from previous one.', metadata={}), Message(role='assistant', text='\n😄 Sure thing! Here’s another one for you:  \n**Why did the scarecrow win an award?**  \nBecause he was outstanding in his field! 🌾😂', metadata={'think_start': '<think>', 'think_text': '\n', 'think_end': '</think>'}), Message(role='user', text='Tell me more complex joke.', metadata={}), Message(role='assistant', text='\n😄 Alright, here

AttributeError: 'HistoryManager' object has no attribute 'get_message_number'